# 07 — Explainability and error analysis of NAFIM
> ⚠️ Figures in this notebook show real (offensive) memes. Set `BLUR_IMAGES = True` for figures you share.

| Part | What | GPU |
|---|---|---|
| 1 | Re-train NAFIM and the mixed-text fusion baseline (seed 42) — notebook 04 kept predictions only | ~30 min |
| 2 | **Modality reliance**: how much each model's prediction changes when the text or the image is removed, by OCR error level | minutes |
| 3 | **Grad-CAM** on the meme image (through CLIP ViT-L/14) + **token attributions** (gradient × input) for selected memes, with NAFIM's reliability score | minutes |
| 4 | **Error analysis** over the 5 notebook-04 seeds: confusion matrices, per-class F1, errors by OCR error level and text length, fixes vs. breaks against the baseline | none |
| 5 | **OCR shortcut check**: do words that OCR reads but the human captions omit (watermarks, outlet names…) predict the label? | none |

**Requires:** notebooks 01–05. Kernel **Python (internship1)**. Runtime ~40 min. Re-running skips finished tasks.

## 1. Imports & config

In [ ]:
import os, re, gc, json, time, random, platform, unicodedata
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageFilter
from tqdm.auto import tqdm
import transformers, sklearn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, CLIPModel, CLIPProcessor
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from rapidfuzz.distance import Levenshtein
import matplotlib.pyplot as plt

ROOT = "/mnt/f415dcc8-27a5-4e66-8493-25f1d939974c/internshipwork/journal"
P, FEAT = f"{ROOT}/processed", f"{ROOT}/features"
NB02, NB03, NB04, NB05 = (f"{ROOT}/results/{x}" for x in ["02_finetuned", "03_ocr_eval", "04_nafim", "05_ablation_crossdataset"])
RES = f"{ROOT}/results/07_xai"
FIG = f"{RES}/figures"
os.makedirs(FIG, exist_ok=True)

DATASETS = ["bhm", "mimosa"]
TASKS = [("bhm", "label"), ("mimosa", "label"), ("mimosa", "label_bin")]
SEEDS = [42, 43, 44, 45, 46]          # notebook 04 seeds (error analysis)
XAI_SEED = 42                         # seed of the re-trained models used for gradients / reliance
TEXT_MODELS = {"BanglaBERT": "csebuetnlp/banglabert", "XLM-R": "xlm-roberta-base"}
CLIP_NAME = "openai/clip-vit-large-patch14"
REAL_SOURCES = {"gold": "text", "surya": "text_surya", "easyocr": "text_easyocr", "ocr": "text_ocr"}
SYNTH_K, SYNTH_SEED = 3, 0
EVAL_SOURCES = ["gold", "surya", "easyocr"]
LAMBDA_IMG = 0.5
HIGH_CER = 0.5
N_PER_GROUP = 2                       # example memes per group (rescued / confident correct / error)
BLUR_IMAGES = True
EVAL_BS = 64
CLASS_NAMES = {("bhm", "label"): ["non-hate", "hate"],
               ("mimosa", "label"): ["non-aggressive", "political", "religious", "gendered", "other"],
               ("mimosa", "label_bin"): ["non-aggressive", "aggressive"]}
CER_BKT = [("CER < 0.2", 0, 0.2), ("0.2 – 0.5", 0.2, 0.5), ("CER ≥ 0.5", 0.5, np.inf)]

dev = "cuda"
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| AMP:", AMP_DTYPE)
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 2**30:.2f} / {total / 2**30:.2f} GiB  (need ~6 GiB: close notebook 06 / other kernels first)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def slug(s):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", s).strip("_")

## 2. Settings inherited from notebooks 02–04 (identical model definition and training)

In [ ]:
cfg02 = json.load(open(f"{NB02}/config.json"))["text"]
cfg03 = json.load(open(f"{NB03}/config.json"))
cfg04 = json.load(open(f"{NB04}/config.json"))
MAX_LEN, BS = cfg02["max_len"], cfg02["batch_size"]
MAX_EPOCHS, PATIENCE = cfg02["max_epochs"], cfg02["patience"]
LR_ENC, LR_HEAD, WD = cfg02["lr_encoder"], cfg02["lr_head"], cfg02["weight_decay"]
WARMUP, DROPOUT = cfg02["warmup"], cfg02["dropout"]
BASE_MODEL = cfg03["selected_model"]
ENC_KEY = BASE_MODEL.split(" ", 1)[1].replace("+CLIP", "")
FREEZE_WORD_EMB = bool(cfg04["word_embeddings_frozen"])
LAMBDA_REL, LAMBDA_CONS = cfg04["nafim"]["lambda_rel"], cfg04["nafim"]["lambda_cons"]
NAFIM_MIX = cfg04["nafim"]["text_source_mix"]
MIXED_MIX = cfg04["mixed_baseline"]["text_source_mix"]
NO_SYNTH_MIX = {"gold": 1 / 3, "surya": 1 / 3, "easyocr": 1 / 3}
print("encoder:", ENC_KEY, "| frozen word emb:", FREEZE_WORD_EMB, "| lambda_rel:", LAMBDA_REL, "| lambda_cons:", LAMBDA_CONS)
print("NAFIM mix:", NAFIM_MIX)

# every model variant trained in this notebook
VARIANTS = {
    "Fusion (mixed text)":        dict(nafim=False, gate=False, lrel=0.0,        lcons=0.0,         mix=MIXED_MIX,    fallback=False),
    "NAFIM":                      dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=False),
    "NAFIM + image fallback":     dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=True),
    "NAFIM w/o gate":             dict(nafim=True,  gate=False, lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=False),
    "NAFIM w/o reliability loss": dict(nafim=True,  gate=True,  lrel=0.0,        lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=False),
    "NAFIM w/o consistency":      dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=0.0,         mix=NAFIM_MIX,    fallback=False),
    "NAFIM w/o synthetic noise":  dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NO_SYNTH_MIX, fallback=False),
}
ABLATIONS = ["NAFIM w/o gate", "NAFIM w/o reliability loss", "NAFIM w/o consistency", "NAFIM w/o synthetic noise"]

## 3. Data (identical to notebooks 04–05)

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[^\u0980-\u09FFa-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def cer(ref, hyp):
    r, h = norm(ref), norm(hyp)
    return np.nan if not r else Levenshtein.distance(r, h) / len(r)

BN_LETTERS = [chr(c) for c in range(0x0985, 0x09BA)]
BN_SIGNS = list("\u09BE\u09BF\u09C0\u09C1\u09C2\u09C3\u09C7\u09C8\u09CB\u09CC\u09CD\u0982\u0983\u0981")
LATIN = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

def corrupt(text, rng):
    rate = rng.uniform(0.05, 0.45)
    out = []
    for ch in str(text):
        u = rng.random()
        if u < rate * 0.30:
            continue
        elif u < rate * 0.60:
            out.append(rng.choice(BN_LETTERS + LATIN))
        elif u < rate * 0.75:
            out.append(ch); out.append(rng.choice(BN_LETTERS + LATIN))
        elif u < rate and ch in BN_SIGNS:
            continue
        else:
            out.append(ch)
    words = "".join(out).split()
    if words and rng.random() < 0.2:
        words.pop(rng.randrange(len(words)))
    if rng.random() < 0.3:
        junk = "".join(rng.choice(LATIN) for _ in range(rng.randint(3, 8)))
        words.insert(rng.randint(0, len(words)), junk)
    return " ".join(words)

def idx_of(df, s):
    return np.where(df["split"].values == s)[0]

data = {}
for ds in DATASETS:
    df = pd.read_csv(f"{P}/{ds}.csv").fillna("")
    for c in REAL_SOURCES.values():
        df[c] = df[c].astype(str)
    rng = random.Random(f"{SYNTH_SEED}-{ds}")
    for k in range(SYNTH_K):
        df[f"text_synth{k}"] = [corrupt(t, rng) for t in df["text"]]
    img = np.load(f"{FEAT}/{ds}_img.npy").astype(np.float32)
    assert len(df) == len(img)
    data[ds] = {"df": df, "img": torch.from_numpy(img)}

SOURCES = list(REAL_SOURCES) + [f"synth{k}" for k in range(SYNTH_K)]
SRC_COL = {**REAL_SOURCES, **{f"synth{k}": f"text_synth{k}" for k in range(SYNTH_K)}}
SRC_IDX = {s: i for i, s in enumerate(SOURCES)}
for ds in DATASETS:
    df = data[ds]["df"]
    for s in SOURCES:
        df[f"cer_{s}"] = 0.0 if s == "gold" else [cer(r, h) for r, h in zip(df["text"], df[SRC_COL[s]])]
    df.loc[df["text"].map(norm) == "", [f"cer_{s}" for s in SOURCES]] = np.nan

In [ ]:
tok = AutoTokenizer.from_pretrained(TEXT_MODELS[ENC_KEY])

def quality_features(texts):
    pieces = [len(x) for x in tok(list(texts), add_special_tokens=False)["input_ids"]]
    rows = []
    for t, npc in zip(texts, pieces):
        t = str(t); n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
        bn = sum("\u0980" <= c <= "\u09FF" for c in t); lat = sum(c.isascii() and c.isalpha() for c in t)
        dig = sum(c.isdigit() for c in t); sp = sum(c.isspace() for c in t)
        pun = sum(c in ".,!?;:'\"-।()" for c in t)
        rows.append([np.log1p(len(t)), np.log1p(len(words)), float(len(t.strip()) == 0),
                     bn / n, lat / n, dig / n, max(n - bn - lat - dig - sp - pun, 0) / n, pun / n,
                     sum(len(w) == 1 for w in words) / nw, np.mean([len(w) for w in words]) if words else 0.0,
                     npc / nw])
    return np.array(rows, dtype=np.float32)

def encode(texts):
    e = tok(list(texts), padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return e["input_ids"], e["attention_mask"]

for ds in DATASETS:
    df = data[ds]["df"]
    enc = [encode(df[SRC_COL[s]]) for s in SOURCES]
    data[ds]["ids"] = torch.stack([e[0] for e in enc]); data[ds]["mask"] = torch.stack([e[1] for e in enc])
    Q = np.stack([quality_features(df[SRC_COL[s]].tolist()) for s in SOURCES])
    tr = idx_of(df, "train")
    train_src = [SRC_IDX[s] for s in ["gold", "surya", "easyocr"]] + [SRC_IDX[f"synth{k}"] for k in range(SYNTH_K)]
    mu = Q[train_src][:, tr].reshape(-1, Q.shape[-1]).mean(0); sd = Q[train_src][:, tr].reshape(-1, Q.shape[-1]).std(0) + 1e-6
    data[ds]["q_mu"], data[ds]["q_sd"] = mu, sd
    data[ds]["q"] = torch.from_numpy((Q - mu) / sd)
    rel = np.stack([np.clip(1 - df[f"cer_{s}"].values, 0, 1) for s in SOURCES])
    data[ds]["rel"] = torch.from_numpy(rel.astype(np.float32))
    print(ds, "ids", tuple(data[ds]["ids"].shape))

## 4. Models (same as notebook 05, plus an `inputs_embeds` path for token attributions)

In [ ]:
def build_encoder():
    enc = AutoModel.from_pretrained(TEXT_MODELS[ENC_KEY])
    if FREEZE_WORD_EMB:
        enc.get_input_embeddings().weight.requires_grad_(False)
    return enc

def mean_pool(hs, mask):
    m = mask.unsqueeze(-1).to(hs.dtype)
    return (hs * m).sum(1) / m.sum(1).clamp(min=1)

class FusionBaseline(nn.Module):
    def __init__(self, n, img_dim):
        super().__init__()
        self.enc = build_encoder(); h = self.enc.config.hidden_size
        self.img_proj = nn.Sequential(nn.Linear(img_dim, h), nn.GELU(), nn.Dropout(DROPOUT))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * h, h), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h, n))
        self.img_logits = None
    def forward(self, ids, mask, img, q=None, inputs_embeds=None):
        hs = self.enc(input_ids=None if inputs_embeds is not None else ids, inputs_embeds=inputs_embeds,
                      attention_mask=mask).last_hidden_state
        t = mean_pool(hs, mask)
        return self.head(torch.cat([t, self.img_proj(img)], -1)), None

class NAFIM(nn.Module):
    """Reliability-gated fusion. With fallback=True the output is a reliability-weighted mixture
    r * p(multimodal) + (1 - r) * p(image only); log-probabilities are returned, so softmax/CE stay valid."""
    def __init__(self, n, img_dim, q_dim, gate=True, fallback=False):
        super().__init__()
        self.enc = build_encoder(); h = self.enc.config.hidden_size
        self.gate, self.fallback = gate, fallback
        self.img_proj = nn.Sequential(nn.Linear(img_dim, h), nn.GELU(), nn.Dropout(DROPOUT))
        self.rel_head = nn.Sequential(nn.Linear(h + q_dim, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * h + 1, h), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h, n))
        if fallback:
            self.img_head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(h, n))
        self.img_logits = None
    def forward(self, ids, mask, img, q, inputs_embeds=None):
        hs = self.enc(input_ids=None if inputs_embeds is not None else ids, inputs_embeds=inputs_embeds,
                      attention_mask=mask).last_hidden_state
        t = mean_pool(hs, mask)
        r = torch.sigmoid(self.rel_head(torch.cat([t, q.to(t.dtype)], -1)))
        tg = r * t if self.gate else t
        iv = self.img_proj(img)
        lo = self.head(torch.cat([tg, iv, r], -1))
        if not self.fallback:
            return lo, r.squeeze(-1)
        li = self.img_head(iv)
        self.img_logits = li
        rf = r.float()
        p = rf * F.softmax(lo.float(), -1) + (1 - rf) * F.softmax(li.float(), -1)
        return torch.log(p.clamp(min=1e-8)), r.squeeze(-1)

## 5. Training (same procedure as notebooks 04–05; returns the trained model)

In [ ]:
def class_weights(y, n):
    counts = np.bincount(y, minlength=n)
    return torch.tensor(len(y) / (n * np.maximum(counts, 1)), dtype=torch.float32, device=dev)

def sample_sources(n_items, mix, g):
    groups = list(mix)
    pick = torch.multinomial(torch.tensor([mix[k] for k in groups]), n_items, replacement=True, generator=g)
    synth = torch.randint(0, SYNTH_K, (n_items,), generator=g)
    return torch.tensor([SRC_IDX[f"synth{int(synth[i])}"] if groups[p] == "synth" else SRC_IDX[groups[p]]
                         for i, p in enumerate(pick.tolist())])

@torch.no_grad()
def _predict(model, ids, mask, img, q):
    model.eval(); probs, rels = [], []
    for i in range(0, len(ids), EVAL_BS):
        sl = slice(i, i + EVAL_BS)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            lo, r = model(ids[sl].to(dev), mask[sl].to(dev), img[sl].to(dev), q[sl].to(dev))
        probs.append(torch.softmax(lo.float(), 1).cpu())
        rels.append(r.float().cpu() if r is not None else torch.full((len(ids[sl]),), float("nan")))
    return torch.cat(probs).numpy(), torch.cat(rels).numpy()

def predict(model, ds, src, idx):
    d = data[ds]; s = SRC_IDX[src]; b = torch.from_numpy(idx)
    return _predict(model, d["ids"][s, b], d["mask"][s, b], d["img"][b], d["q"][s, b])

def run_train(name, ds, lab, seed, ckpt=False, cross=False):
    v = VARIANTS[name]
    set_seed(seed)
    d = data[ds]; df = d["df"]; y = df[lab].values.astype(np.int64); n = int(y.max()) + 1
    Y = torch.from_numpy(y)
    tr, va, te = [idx_of(df, s) for s in ("train", "val", "test")]
    if v["nafim"]:
        model = NAFIM(n, d["img"].shape[1], d["q"].shape[-1], gate=v["gate"], fallback=v["fallback"]).to(dev)
    else:
        model = FusionBaseline(n, d["img"].shape[1]).to(dev)
    if ckpt:
        model.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    enc_ids = {id(p) for p in model.enc.parameters()}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    enc_params = [p for p in model.enc.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENC}, {"params": head_params, "lr": LR_HEAD}],
                            weight_decay=WD, foreach=False)
    spe = (len(tr) + BS - 1) // BS
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * spe * MAX_EPOCHS), spe * MAX_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    cw = class_weights(y[tr], n)
    g = torch.Generator().manual_seed(seed)
    trainable = [p for p in model.parameters() if p.requires_grad]

    best_f, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.from_numpy(tr)[torch.randperm(len(tr), generator=g)]
        for i in tqdm(range(0, len(perm), BS), desc=f"{slug(name)} {ds}/{lab} s{seed} ep{ep + 1}", leave=False):
            b = perm[i:i + BS]; src = sample_sources(len(b), v["mix"], g); yb = Y[b].to(dev)
            if v["nafim"]:
                gold = torch.full_like(src, SRC_IDX["gold"])
                s2, b2 = torch.cat([gold, src]), torch.cat([b, b])
                ids, mask = d["ids"][s2, b2].to(dev), d["mask"][s2, b2].to(dev)
                img, q, rt = d["img"][b2].to(dev), d["q"][s2, b2].to(dev), d["rel"][s2, b2].to(dev)
            else:
                ids, mask = d["ids"][src, b].to(dev), d["mask"][src, b].to(dev)
                img, q = d["img"][b].to(dev), d["q"][src, b].to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                lo, r = model(ids, mask, img, q)
            lo = lo.float()
            if v["nafim"]:
                lg, lv = lo[:len(b)], lo[len(b):]
                yy = torch.cat([yb, yb])
                loss = 0.5 * (F.cross_entropy(lg, yb, weight=cw) + F.cross_entropy(lv, yb, weight=cw))
                if v["lrel"] > 0:
                    ok = ~torch.isnan(rt)
                    if ok.any():
                        loss = loss + v["lrel"] * F.mse_loss(r.float()[ok], rt[ok])
                if v["lcons"] > 0:
                    loss = loss + v["lcons"] * F.kl_div(F.log_softmax(lv, -1), F.softmax(lg, -1).detach(), reduction="batchmean")
                if v["fallback"]:
                    loss = loss + LAMBDA_IMG * F.cross_entropy(model.img_logits.float(), yy, weight=cw)
            else:
                loss = F.cross_entropy(lo, yb, weight=cw)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        f = np.mean([f1_score(y[va], predict(model, ds, s, va)[0].argmax(1), average="macro") for s in EVAL_SOURCES])
        if f > best_f:
            best_f, best_ep, bad = f, ep + 1, 0
            best_state = {k: v_.detach().cpu().clone() for k, v_ in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    out = {s: predict(model, ds, s, te) for s in REAL_SOURCES}
    del opt, best_state, sch, scaler
    gc.collect(); torch.cuda.empty_cache()
    return model, out, y[te]

## 6. Helpers: modality reliance, Grad-CAM through CLIP, token attributions

In [ ]:
def bucket_of(c):
    if np.isnan(c):
        return "no gold text"
    for name, lo, hi in CER_BKT:
        if lo <= c < hi:
            return name

def tv(p, q):
    return 0.5 * np.abs(p - q).sum(1)

def modality_reliance(model, ds, src, idx):
    """Total-variation change of the predicted distribution when the text is removed (empty string)
    or the image is replaced by the mean training image feature."""
    d = data[ds]; s = SRC_IDX[src]; b = torch.from_numpy(idx)
    ids, mask, img, q = d["ids"][s, b], d["mask"][s, b], d["img"][b], d["q"][s, b]
    p_full = _predict(model, ids, mask, img, q)[0]
    e_ids, e_mask = encode([""] * len(idx))
    e_q = torch.from_numpy(((quality_features([""] * len(idx)) - d["q_mu"]) / d["q_sd"]).astype(np.float32))
    p_notext = _predict(model, e_ids, e_mask, img, e_q)[0]
    mean_img = d["img"][idx_of(d["df"], "train")].mean(0, keepdim=True).expand(len(idx), -1)
    p_noimg = _predict(model, ids, mask, mean_img, q)[0]
    return tv(p_full, p_notext), tv(p_full, p_noimg), p_full

clip_model = CLIPModel.from_pretrained(CLIP_NAME).eval()      # kept on CPU; moved to the GPU only for Grad-CAM
for p_ in clip_model.parameters():
    p_.requires_grad_(False)
clip_proc = CLIPProcessor.from_pretrained(CLIP_NAME)

def load_image(path):
    img = Image.open(path).convert("RGB")
    return img.filter(ImageFilter.GaussianBlur(12)) if BLUR_IMAGES else img

def merge_word_pieces(tokens, scores):
    words, vals = [], []
    specials = set(tok.all_special_tokens)
    sentencepiece = any(t.startswith("\u2581") for t in tokens)      # XLM-R style; else WordPiece (##)
    for t, v in zip(tokens, scores):
        if t in specials:
            continue
        cont = t.startswith("##") or (sentencepiece and not t.startswith("\u2581"))
        if cont and words:
            words[-1] += t[2:] if t.startswith("##") else t; vals[-1] += v
        else:
            words.append(t.lstrip("\u2581")); vals.append(v)
    keep = [(w, v) for w, v in zip(words, vals) if w.strip()]
    return [w for w, _ in keep], np.array([v for _, v in keep])

def explain(model, ds, i, src):
    """Grad-CAM on CLIP layer -2 patch tokens + gradient×input token attributions, for NAFIM's predicted class."""
    d = data[ds]; df = d["df"]; s = SRC_IDX[src]
    img = Image.open(df["image_path"].iloc[i]).convert("RGB")
    pix = clip_proc(images=img, return_tensors="pt")["pixel_values"].to(dev).requires_grad_(True)
    acts = {}
    h = clip_model.vision_model.encoder.layers[-2].register_forward_hook(lambda m, inp, out: acts.__setitem__("a", out[0]))
    vout = clip_model.vision_model(pixel_values=pix)
    h.remove(); acts["a"].retain_grad()
    feat = F.normalize(clip_model.visual_projection(vout.pooler_output), dim=-1)
    cos = F.cosine_similarity(feat.detach().cpu().float(), d["img"][i][None]).item()

    ids, mask, q = d["ids"][s, i][None].to(dev), d["mask"][s, i][None].to(dev), d["q"][s, i][None].to(dev)
    emb = model.enc.get_input_embeddings()(ids).detach().requires_grad_(True)
    model.eval()
    lo, r = model(None, mask, feat, q, inputs_embeds=emb)
    probs = torch.softmax(lo.float(), -1)[0]; pred = int(probs.argmax())
    lo[0, pred].backward()

    a, g = acts["a"][0, 1:].detach(), acts["a"].grad[0, 1:]
    cam = torch.relu((a * g.mean(0)).sum(-1))
    side = int(round(cam.numel() ** 0.5)); cam = cam[: side * side].reshape(side, side)
    cam = (cam / cam.max().clamp(min=1e-8)).cpu().numpy()

    m = mask[0].bool()
    attr = (emb.grad[0] * emb[0]).sum(-1)[m].detach().float().cpu().numpy()
    words, wattr = merge_word_pieces(tok.convert_ids_to_tokens(ids[0][m].tolist()), attr)
    return {"pred": pred, "prob": float(probs[pred]), "r": float(r) if r is not None else np.nan,
            "cam": cam, "words": words, "word_attr": wattr, "clip_cos": cos}

def top_words(words, attr, k=5):
    if len(words) == 0:
        return ""
    o = np.argsort(-np.abs(attr))[:k]
    return " | ".join(f"{words[j]} ({attr[j]:+.3f})" for j in o)

In [ ]:
def plot_xai(figs, out, per_group=1):
    """figs: list of (image, cam, row). One column per example: meme on top, Grad-CAM below."""
    seen, sel = {}, []
    for f in figs:
        g = f[2]["group"]
        if seen.get(g, 0) < per_group:
            sel.append(f); seen[g] = seen.get(g, 0) + 1
    n = len(sel)
    fig, axes = plt.subplots(2, n, figsize=(2.3 * n, 4.9), squeeze=False)
    for j, (img, cam, r_) in enumerate(sel):
        img = img if isinstance(img, Image.Image) else Image.fromarray(np.asarray(img, dtype=np.uint8))
        im = np.array(img.convert("RGB").resize((224, 224)))
        heat = np.array(Image.fromarray((np.asarray(cam) * 255).astype(np.uint8))
                        .resize((224, 224), Image.BILINEAR)) / 255
        axes[0, j].imshow(im)
        axes[1, j].imshow(im); axes[1, j].imshow(heat, cmap="jet", alpha=0.45)
        axes[0, j].set_title(f"{r_['group']}\ntrue: {r_['true']}\nNAFIM: {r_['nafim_pred']} "
                             f"({r_['nafim_prob']:.2f}) | base: {r_['baseline_pred']}", fontsize=6)
        axes[1, j].set_title(f"r = {r_['reliability_r']:.2f} | CER = {r_['true_cer_ocr']:.2f}", fontsize=6)
        for a in axes[:, j]: a.axis("off")
    fig.tight_layout(pad=0.3); fig.savefig(out, dpi=300, bbox_inches="tight"); plt.close(fig)

## 7. Per task: re-train, modality reliance, explained examples (resumable)

In [ ]:
REL_CSV, EX_CSV = f"{RES}/modality_reliance.csv", f"{RES}/xai_examples.csv"
rel_all = pd.read_csv(REL_CSV) if os.path.exists(REL_CSV) else pd.DataFrame(columns=["dataset", "task"])
ex_all = pd.read_csv(EX_CSV) if os.path.exists(EX_CSV) else pd.DataFrame(columns=["dataset", "task"])
done_tasks = set(zip(rel_all["dataset"], rel_all["task"])) & set(zip(ex_all["dataset"], ex_all["task"]))

for ds, lab in TASKS:
    if (ds, lab) in done_tasks:
        print("done:", ds, lab); continue
    df = data[ds]["df"]; te = idx_of(df, "test"); cer_te = df["cer_ocr"].values[te]
    models, outs = {}, {}
    for name in ["NAFIM", "Fusion (mixed text)"]:
        t0 = time.time()
        models[name], outs[name], y = run_train(name, ds, lab, XAI_SEED)
        for p_ in models[name].parameters():
            p_.requires_grad_(False)
        models[name].to("cpu"); gc.collect(); torch.cuda.empty_cache()   # free the GPU for the next model
        print(f"{ds}/{lab} {name}: OCR-text test F1 = {f1_score(y, outs[name]['ocr'][0].argmax(1), average='macro'):.4f}"
              f"  ({(time.time() - t0) / 60:.1f} min)")

    rows = []
    for name, m in models.items():
        m.to(dev)
        for src in ["gold", "ocr"]:
            t_dep, i_dep, p = modality_reliance(m, ds, src, te)
            rows.append(pd.DataFrame({"dataset": ds, "task": lab, "model": name, "test_text": src, "row": te,
                                      "cer_ocr": cer_te, "bucket": [bucket_of(c) for c in cer_te],
                                      "text_dependence": t_dep, "image_dependence": i_dep,
                                      "correct": p.argmax(1) == y}))
        m.to("cpu"); gc.collect(); torch.cuda.empty_cache()
    rel_task = pd.concat(rows)

    pn, pm = outs["NAFIM"]["ocr"][0], outs["Fusion (mixed text)"]["ocr"][0]
    cn, cm_ = pn.argmax(1) == y, pm.argmax(1) == y; conf = pn.max(1)
    hi = np.nan_to_num(cer_te, nan=0) >= HIGH_CER
    groups = {"rescued (NAFIM right, baseline wrong)": np.where(cn & ~cm_ & hi)[0],
              "confident correct (harmful)": np.where(cn & (y > 0))[0],
              "NAFIM error": np.where(~cn)[0]}
    if len(groups["rescued (NAFIM right, baseline wrong)"]) < N_PER_GROUP:
        groups["rescued (NAFIM right, baseline wrong)"] = np.where(cn & ~cm_)[0]
    ex_rows, figs = [], []
    models["NAFIM"].to(dev); clip_model.to(dev)
    for gname, cand in groups.items():
        for j in cand[np.argsort(-conf[cand])][:N_PER_GROUP]:
            i = te[j]; e = explain(models["NAFIM"], ds, i, "ocr")
            ex_rows.append({"dataset": ds, "task": lab, "group": gname, "row": i, "image_path": df["image_path"].iloc[i],
                            "true": CLASS_NAMES[(ds, lab)][y[j]], "nafim_pred": CLASS_NAMES[(ds, lab)][e["pred"]],
                            "nafim_prob": e["prob"], "baseline_pred": CLASS_NAMES[(ds, lab)][int(pm[j].argmax())],
                            "reliability_r": e["r"], "true_cer_ocr": cer_te[j], "clip_feature_cos": e["clip_cos"],
                            "gold_text": df["text"].iloc[i], "ocr_text": df["text_ocr"].iloc[i],
                            "top_words_ocr": top_words(e["words"], e["word_attr"])})
            figs.append((load_image(df["image_path"].iloc[i]), e["cam"], ex_rows[-1]))

    models["NAFIM"].to("cpu"); clip_model.to("cpu"); gc.collect(); torch.cuda.empty_cache()
    # if figs:
    #     fig, axes = plt.subplots(len(figs), 2, figsize=(6.4, 3.0 * len(figs)))
    #     axes = np.atleast_2d(axes)
    #     for (img, cam, r_), (a0, a1) in zip(figs, axes):
    #         a0.imshow(img); a0.axis("off")
    #         a0.set_title(f"{r_['group']}\ntrue: {r_['true']} | NAFIM: {r_['nafim_pred']} ({r_['nafim_prob']:.2f}) | base: {r_['baseline_pred']}", fontsize=7)
    #         a1.imshow(img.resize((224, 224))); a1.imshow(np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255,
    #                                                     cmap="jet", alpha=0.45); a1.axis("off")
    #         a1.set_title(f"Grad-CAM | r = {r_['reliability_r']:.2f} | true CER = {r_['true_cer_ocr']:.2f}", fontsize=7)
    #     fig.tight_layout(); fig.savefig(f"{FIG}/xai_examples_{ds}_{lab}.png", dpi=300, bbox_inches="tight"); plt.close(fig)
    
    if figs:
    # save images + cams so the figure can be re-plotted later without retraining
        np.savez_compressed(f"{FIG}/cams_{ds}_{lab}.npz",
            imgs=np.stack([np.array(i.convert("RGB").resize((224, 224))) for i, _, _ in figs]),
            cams=np.stack([c for _, c, _ in figs]))
        pd.DataFrame([r_ for *_, r_ in figs]).to_csv(f"{FIG}/cams_{ds}_{lab}_rows.csv", index=False)
        plot_xai(figs, f"{FIG}/xai_examples_{ds}_{lab}.png")

    rel_all = pd.concat([rel_all[~((rel_all.dataset == ds) & (rel_all.task == lab))], rel_task]); rel_all.to_csv(REL_CSV, index=False)
    ex_all = pd.concat([ex_all[~((ex_all.dataset == ds) & (ex_all.task == lab))], pd.DataFrame(ex_rows)]); ex_all.to_csv(EX_CSV, index=False, encoding="utf-8")
    del models; gc.collect(); torch.cuda.empty_cache()
    print("saved", ds, lab)
print("CLIP feature check (should be ~1.0):", ex_all["clip_feature_cos"].astype(float).round(3).describe()[["mean", "min"]].to_dict())

## 8. Modality reliance — does NAFIM lean on the text less when OCR is bad?

In [ ]:
rel_all = pd.read_csv(REL_CSV)
order_b = [b for b, _, _ in CER_BKT]
rel_tab = (rel_all[rel_all.test_text == "ocr"].groupby(["dataset", "task", "model", "bucket"])
           [["text_dependence", "image_dependence"]].mean().round(3)
           .unstack("bucket").reindex(columns=order_b, level=1))
display(rel_tab)

fig, axes = plt.subplots(1, len(TASKS), figsize=(4.6 * len(TASKS), 3.6))
for ax, (ds, lab) in zip(np.atleast_1d(axes), TASKS):
    sub = rel_all[(rel_all.dataset == ds) & (rel_all.task == lab) & (rel_all.test_text == "ocr")]
    for name, st in [("NAFIM", "-o"), ("Fusion (mixed text)", "--s")]:
        v = sub[sub.model == name].groupby("bucket")["text_dependence"].mean().reindex(order_b)
        ax.plot(order_b, v.values, st, label=name)
    ax.set_title(f"{ds}/{lab}"); ax.grid(alpha=0.3); ax.tick_params(axis="x", labelsize=8)
np.atleast_1d(axes)[0].set_ylabel("text dependence")
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.suptitle(f"Reliance on (OCR) text by OCR error level — seed {XAI_SEED}", fontsize=10)
fig.tight_layout(); fig.savefig(f"{FIG}/text_dependence_by_cer.png", dpi=300, bbox_inches="tight"); plt.show()

## 9. Explained examples (Grad-CAM figures are in `figures/xai_examples_*.png`)

In [ ]:
ex_all = pd.read_csv(EX_CSV)
show = ["dataset", "task", "group", "true", "nafim_pred", "nafim_prob", "baseline_pred", "reliability_r", "true_cer_ocr", "top_words_ocr"]
ex_all[show].round(3)

## 10. Error analysis over the 5 notebook-04 seeds (majority vote, OCR text)

In [ ]:
def nb04_slug(s):
    return s.replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")

def vote(model, ds, lab, src="ocr"):
    P_ = [np.load(f).argmax(1) for s in SEEDS
          if os.path.exists(f := f"{NB04}/preds/{ds}__{lab}__{nb04_slug(model)}__s{s}__test-{src}.npy")]
    if not P_:
        return None
    P_ = np.stack(P_); n = P_.max() + 1
    return np.array([np.bincount(col, minlength=n).argmax() for col in P_.T])

err_rows, fix_rows, rep_rows = [], [], []
fig, axes = plt.subplots(1, len(TASKS), figsize=(4.4 * len(TASKS), 3.9))
for ax, (ds, lab) in zip(np.atleast_1d(axes), TASKS):
    df = data[ds]["df"]; te = idx_of(df, "test"); y = df[lab].values[te].astype(int)
    pn, pb = vote("NAFIM", ds, lab), vote("Fusion (mixed text)", ds, lab)
    names = CLASS_NAMES[(ds, lab)]
    cm = confusion_matrix(y, pn, labels=range(len(names)), normalize="true")
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for a_ in range(len(names)):
        for b_ in range(len(names)):
            ax.text(b_, a_, f"{cm[a_, b_]:.2f}", ha="center", va="center", fontsize=7, color="w" if cm[a_, b_] > 0.5 else "k")
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=35, ha="right", fontsize=7)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
    ax.set_title(f"NAFIM — {ds}/{lab}", fontsize=9); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    rep = classification_report(y, pn, labels=range(len(names)), target_names=names, output_dict=True, zero_division=0)
    for c in names:
        rep_rows.append({"dataset": ds, "task": lab, "class": c, "precision": rep[c]["precision"],
                         "recall": rep[c]["recall"], "f1": rep[c]["f1-score"], "support": rep[c]["support"]})
    cer = df["cer_ocr"].values[te]; nw = df["text_ocr"].str.split().str.len().values[te]
    lenb = pd.cut(nw, [-1, 0, 5, 15, 10**6], labels=["empty", "1–5 words", "6–15 words", ">15 words"])
    for model, p_ in [("NAFIM", pn), ("Fusion (mixed text)", pb)]:
        for b in order_b + ["no gold text"]:
            m = np.array([bucket_of(c) == b for c in cer])
            if m.sum():
                err_rows.append({"dataset": ds, "task": lab, "model": model, "by": "OCR error", "group": b,
                                 "n": int(m.sum()), "error_rate": float((p_[m] != y[m]).mean())})
        for b in lenb.categories:
            m = np.asarray(lenb == b)
            if m.sum():
                err_rows.append({"dataset": ds, "task": lab, "model": model, "by": "OCR text length", "group": b,
                                 "n": int(m.sum()), "error_rate": float((p_[m] != y[m]).mean())})
    fix_rows.append({"dataset": ds, "task": lab, "NAFIM fixes (base wrong → NAFIM right)": int(((pb != y) & (pn == y)).sum()),
                     "NAFIM breaks (base right → NAFIM wrong)": int(((pb == y) & (pn != y)).sum()),
                     "both wrong": int(((pb != y) & (pn != y)).sum()), "n_test": len(y)})
fig.tight_layout(); fig.savefig(f"{FIG}/confusion_nafim.png", dpi=300, bbox_inches="tight"); plt.show()
err_df, fix_df, rep_df = pd.DataFrame(err_rows), pd.DataFrame(fix_rows), pd.DataFrame(rep_rows)
display(rep_df.round(3)); display(fix_df)
err_df.pivot_table(index=["dataset", "task", "by", "group"], columns="model", values="error_rate").round(3)

## 11. OCR shortcut check — do words OCR reads but the human captions omit predict the label?

In [ ]:
def words(s):
    return [w for w in norm(s).split() if len(w) >= 2]

sc_rows, top_rows = [], []
for ds, lab in [("bhm", "label"), ("mimosa", "label")]:
    df = data[ds]["df"].copy(); y = df[lab].values.astype(int)
    df["extra"] = [" ".join(sorted(set(words(o)) - set(words(g)))) for o, g in zip(df["text_ocr"], df["text"])]
    tr, te = idx_of(df, "train"), idx_of(df, "test")
    names = CLASS_NAMES[(ds, lab)]
    for feat_name, col in [("words only OCR reads (extra)", "extra"), ("gold caption words", "text"), ("OCR text words", "text_ocr")]:
        vec = CountVectorizer(tokenizer=words, lowercase=False, binary=True, min_df=3, token_pattern=None)
        try:
            Xtr = vec.fit_transform(df[col].values[tr]); Xte = vec.transform(df[col].values[te])
        except ValueError:                       # no word appears in >= 3 training memes
            sc_rows.append({"dataset": ds, "task": lab, "features": feat_name, "vocab": 0, "test_macro_f1": np.nan})
            continue
        clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5).fit(Xtr, y[tr])
        sc_rows.append({"dataset": ds, "task": lab, "features": feat_name, "vocab": Xtr.shape[1],
                        "test_macro_f1": f1_score(y[te], clf.predict(Xte), average="macro")})
        if col == "extra":
            ch, _ = chi2(Xtr, y[tr]); voc = np.array(vec.get_feature_names_out())
            for k in np.argsort(-np.nan_to_num(ch))[:15]:
                cnt = np.bincount(y[tr][np.asarray(Xtr[:, k].todense()).ravel() > 0], minlength=len(names))
                top_rows.append({"dataset": ds, "word": voc[k], "chi2": ch[k], "n_train": int(cnt.sum()),
                                 **{f"% {c}": round(100 * v / max(cnt.sum(), 1), 1) for c, v in zip(names, cnt)}})
    has_extra = df["extra"].values[te] != ""
    pn = vote("NAFIM", ds, lab)
    for flag, m in [("with extra OCR words", has_extra), ("without", ~has_extra)]:
        sc_rows.append({"dataset": ds, "task": lab, "features": f"NAFIM accuracy — memes {flag}", "vocab": int(m.sum()),
                        "test_macro_f1": f1_score(y[te][m], pn[m], average="macro") if m.sum() else np.nan})
    sc_rows.append({"dataset": ds, "task": lab, "features": "majority-class baseline", "vocab": 0,
                    "test_macro_f1": f1_score(y[te], np.full(len(te), np.bincount(y[tr]).argmax()), average="macro")})
sc_df, top_df = pd.DataFrame(sc_rows), pd.DataFrame(top_rows)
display(sc_df.round(4))
top_df.round(2)

## 12. Save results + config

In [ ]:
rel_tab.to_csv(f"{RES}/modality_reliance_summary.csv")
err_df.to_csv(f"{RES}/error_by_group.csv", index=False); fix_df.to_csv(f"{RES}/fixes_vs_breaks.csv", index=False)
rep_df.to_csv(f"{RES}/per_class_f1_nafim.csv", index=False)
sc_df.to_csv(f"{RES}/shortcut_check.csv", index=False); top_df.to_csv(f"{RES}/shortcut_top_extra_words.csv", index=False, encoding="utf-8")
config = {"xai_seed": XAI_SEED, "error_analysis_seeds": SEEDS, "error_analysis_vote": "majority over seeds, OCR text",
          "grad_cam": f"{CLIP_NAME}, gradients of NAFIM's predicted-class logit w.r.t. patch tokens of vision layer -2",
          "token_attribution": "gradient x input on word embeddings, summed over hidden dim, merged to words",
          "modality_reliance": "total-variation change of predicted distribution: text -> empty string; image -> mean train CLIP feature",
          "shortcut_check": "logistic regression (balanced, C=0.5) on binary bag of words, min_df=3; extra = OCR words not in gold caption",
          "blur_images": BLUR_IMAGES, "high_cer": HIGH_CER,
          "versions": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
                       "sklearn": sklearn.__version__, "gpu": torch.cuda.get_device_name(0)}}
with open(f"{RES}/config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("saved to", RES)